In [ ]:
import os
import sys
from collections import defaultdict
from copy import deepcopy
from functools import partial
from typing import Dict, List
from warnings import warn

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from matplotlib import pyplot as plt
from mpl_toolkits import axes_grid1
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

sys.path.append("your_path")
from src.common.data.base import Dataframe
from src.tasks.pretrain.encoder import SSLEncoder

# Set random seeds for reproducibility
torch.manual_seed(123)
np.random.seed(123)

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

## CKA

In [ ]:
def add_colorbar(im, aspect=10, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


class CKA_FreqMAE:
    def __init__(
        self,
        model1: nn.Module,
        model2: nn.Module,
        model1_name: str = None,
        model2_name: str = None,
        model1_layers: List[str] = None,
        model2_layers: List[str] = None,
        device: str = "cpu",
    ):
        """

        :param model1: (nn.Module) Neural Network 1
        :param model2: (nn.Module) Neural Network 2
        :param model1_name: (str) Name of model 1
        :param model2_name: (str) Name of model 2
        :param model1_layers: (List) List of layers to extract features from
        :param model2_layers: (List) List of layers to extract features from
        :param device: Device to run the model
        """

        self.model1 = model1
        self.model2 = model2
        self.model1.eval()
        self.model2.eval()

        self.device = device

        self.model1_info = {}
        self.model2_info = {}

        if model1_name is None:
            self.model1_info["Name"] = model1.__repr__().split("(")[0]
        else:
            self.model1_info["Name"] = model1_name

        if model2_name is None:
            self.model2_info["Name"] = model2.__repr__().split("(")[0]
        else:
            self.model2_info["Name"] = model2_name

        if self.model1_info["Name"] == self.model2_info["Name"]:
            warn(
                f"Both model have identical names - {self.model2_info['Name']}. "
                "It may cause confusion when interpreting the results. "
                "Consider giving unique names to the models :)"
            )

        self.model1_info["Layers"] = []
        self.model2_info["Layers"] = []

        self.model1_features = {}
        self.model2_features = {}

        if len(list(model1.modules())) > 150 and model1_layers is None:
            warn(
                "Model 1 seems to have a lot of layers. "
                "Consider giving a list of layers whose features you are concerned with "
                "through the 'model1_layers' parameter. Your CPU/GPU will thank you :)"
            )

        self.model1_layers = model1_layers

        if len(list(model2.modules())) > 150 and model2_layers is None:
            warn(
                "Model 2 seems to have a lot of layers. "
                "Consider giving a list of layers whose features you are concerned with "
                "through the 'model2_layers' parameter. Your CPU/GPU will thank you :)"
            )

        self.model2_layers = model2_layers

        self._insert_hooks()
        self.model1 = self.model1.to(self.device)
        self.model2 = self.model2.to(self.device)

        self.model1.eval()
        self.model2.eval()

    def _log_layer(
        self,
        model: str,
        name: str,
        layer: nn.Module,
        inp: torch.Tensor,
        out: torch.Tensor,
    ):

        if model == "model1":
            self.model1_features[name] = out

        elif model == "model2":
            self.model2_features[name] = out

        else:
            raise RuntimeError("Unknown model name for _log_layer.")

    def _insert_hooks(self):
        # Model 1
        for name, layer in self.model1.named_modules():
            if self.model1_layers is not None:
                if name in self.model1_layers:
                    self.model1_info["Layers"] += [name]
                    layer.register_forward_hook(partial(self._log_layer, "model1", name))
            else:
                self.model1_info["Layers"] += [name]
                layer.register_forward_hook(partial(self._log_layer, "model1", name))

        # Model 2
        for name, layer in self.model2.named_modules():
            if self.model2_layers is not None:
                if name in self.model2_layers:
                    self.model2_info["Layers"] += [name]
                    layer.register_forward_hook(partial(self._log_layer, "model2", name))
            else:

                self.model2_info["Layers"] += [name]
                layer.register_forward_hook(partial(self._log_layer, "model2", name))

    def _HSIC(self, K, L):
        """
        Computes the unbiased estimate of HSIC metric.

        Reference: https://arxiv.org/pdf/2010.15327.pdf Eq (3)
        """
        Kc = self._center_gram(K)
        Lc = self._center_gram(L)
        hsic_val = (Kc * Lc).sum()  # trace of dot product
        n = K.shape[0]
        return hsic_val / ((n - 1) ** 2)

    def _center_gram(self, K: torch.Tensor) -> torch.Tensor:
        """
        Gram matrix K를 평균 0이 되도록 centering.
        H = I - 1/n * 11^T를 사용한 double centering 수행.
        """
        n = K.size(0)
        unit = torch.ones(n, n, device=K.device) / n
        H = torch.eye(n, device=K.device) - unit
        return H @ K @ H  # centered Gram matrix

    @torch.no_grad()
    def compare(
        self,
        dataloader1: DataLoader,
        dataloader2: DataLoader = None,
        high_freq_keep_ratio: float = 1,
        low_freq_keep_ratio: float = 1,
    ) -> None:
        """
        Computes the feature similarity between the models on the
        given datasets.
        :param dataloader1: (DataLoader)
        :param dataloader2: (DataLoader) If given, model 2 will run on this
                            dataset. (default = None)
        """

        if dataloader2 is None:
            warn("Dataloader for Model 2 is not given. Using the same dataloader for both models.")
            dataloader2 = dataloader1

        self.model1_info["Dataset"] = dataloader1.dataset.__repr__().split("\n")[0]
        self.model2_info["Dataset"] = dataloader2.dataset.__repr__().split("\n")[0]

        N = (
            len(self.model1_layers)
            if self.model1_layers is not None
            else len(list(self.model1.modules()))
        )
        M = (
            len(self.model2_layers)
            if self.model2_layers is not None
            else len(list(self.model2.modules()))
        )

        self.hsic_matrix = torch.zeros(N, M, 3, device=self.device)

        num_batches = min(len(dataloader1), len(dataloader1))

        for x1, x2 in tqdm(
            zip(dataloader1, dataloader2),
            desc="| Comparing features |",
            total=num_batches,
        ):

            self.model1_features = {}
            self.model2_features = {}
            _ = self.model1.forward_features(x1.to(self.device))
            _ = self.model2.forward_features(
                x2.to(self.device),
                high_freq_keep_ratio=high_freq_keep_ratio,
                low_freq_keep_ratio=low_freq_keep_ratio,
                fixed_mask=True,
            )

            for i, (name1, feat1) in enumerate(self.model1_features.items()):
                # feat1 = feat1[:, :1]
                feat1 = feat1[:, : self.model1.num_prefix_tokens].flatten(1)
                X = feat1.flatten(1)
                K = X @ X.t()
                self.hsic_matrix[i, :, 0] += self._HSIC(K, K) / num_batches

                for j, (name2, feat2) in enumerate(self.model2_features.items()):
                    # feat2 = feat2[:, :1]
                    feat2 = feat2[:, : self.model2.num_prefix_tokens].flatten(1)
                    Y = feat2.flatten(1)
                    L = Y @ Y.t()
                    assert K.shape == L.shape, f"Feature shape mistach! {K.shape}, {L.shape}"
                    if feat1.shape == feat2.shape:
                        assert not torch.all(
                            torch.eq(feat1, feat2)
                        ), f"Features are identical for {name1} and {name2}"

                    self.hsic_matrix[i, j, 1] += self._HSIC(K, L) / num_batches
                    self.hsic_matrix[i, j, 2] += self._HSIC(L, L) / num_batches

        self.hsic_matrix = self.hsic_matrix[:, :, 1] / (
            self.hsic_matrix[:, :, 0].clamp(min=1e-12).sqrt()
            * self.hsic_matrix[:, :, 2].clamp(min=1e-12).sqrt()
        )
        self.hsic_matrix = self.hsic_matrix.to("cpu")

        assert not torch.isnan(self.hsic_matrix).any(), "HSIC computation resulted in NANs"

    def export(self) -> Dict:
        """
        Exports the CKA data along with the respective model layer names.
        :return:
        """
        return {
            "model1_name": self.model1_info["Name"],
            "model2_name": self.model2_info["Name"],
            "CKA": self.hsic_matrix,
            "model1_layers": self.model1_info["Layers"],
            "model2_layers": self.model2_info["Layers"],
            "dataset1_name": self.model1_info["Dataset"],
            "dataset2_name": self.model2_info["Dataset"],
        }

    def plot_cka_matrix(self, save_path: str = None, title: str = None):
        fig, ax = plt.subplots()
        im = ax.imshow(self.hsic_matrix, origin="lower", cmap="magma", vmin=0, vmax=1)
        ax.set_xlabel(f"Layers {self.model2_info['Name']}", fontsize=15)
        ax.set_ylabel(f"Layers {self.model1_info['Name']}", fontsize=15)

        if title is not None:
            ax.set_title(f"{title}", fontsize=18, pad=15)
        else:
            ax.set_title(f"{self.model1_info['Name']} vs {self.model2_info['Name']}", fontsize=18)

        add_colorbar(im)
        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=300)

        plt.show()

    def plot_linear_cka(self, save_path: str = None, title: str = None):
        # Extract diagonal elements from the HSIC matrix (standard diagonal)
        n = self.hsic_matrix.shape[0]
        diagonal_cka = torch.diag(self.hsic_matrix)

        fig, ax = plt.subplots()
        ax.plot(diagonal_cka, marker="o", linestyle="-", color="b")
        ax.set_xlabel(f"Layers {self.model1_info['Name']}", fontsize=15)
        ax.set_ylabel("CKA", fontsize=15)
        # Set y-axis range from 0 to 1
        ax.set_ylim(0, 1.02)
        # Set x-axis ticks to show block numbers
        num_layers = len(diagonal_cka)
        ax.set_xticks(range(num_layers))
        ax.set_xticklabels([f"block_{i}" for i in range(num_layers)], rotation=45)

        if title is not None:
            ax.set_title(f"{title}", fontsize=18, pad=15)
        else:
            ax.set_title(
                f"{self.model1_info['Name']} vs {self.model2_info['Name']} (Diagonal CKA)",
                fontsize=18,
                pad=15,
            )

        if save_path is not None:
            plt.savefig(save_path, dpi=300)

        plt.show()


class CKA_MAE(CKA_FreqMAE):
    @torch.no_grad()
    def compare(
        self,
        dataloader1: DataLoader,
        dataloader2: DataLoader = None,
        pred_ratio: float = 1,
    ) -> None:
        if dataloader2 is None:
            warn("Dataloader for Model 2 is not given. Using the same dataloader for both models.")
            dataloader2 = dataloader1

        self.model1_info["Dataset"] = dataloader1.dataset.__repr__().split("\n")[0]
        self.model2_info["Dataset"] = dataloader2.dataset.__repr__().split("\n")[0]

        N = (
            len(self.model1_layers)
            if self.model1_layers is not None
            else len(list(self.model1.modules()))
        )
        M = (
            len(self.model2_layers)
            if self.model2_layers is not None
            else len(list(self.model2.modules()))
        )

        self.hsic_matrix = torch.zeros(N, M, 3, device=self.device)

        num_batches = min(len(dataloader1), len(dataloader1))

        for x1, x2 in tqdm(
            zip(dataloader1, dataloader2),
            desc="| Comparing features |",
            total=num_batches,
        ):

            self.model1_features = {}
            self.model2_features = {}
            _ = self.model1.forward_features(x1.to(self.device))
            _ = self.model2.forward_features(
                x2.to(self.device),
                mask_ratio=1 - pred_ratio,
                fixed_mask=True,
            )

            for i, (name1, feat1) in enumerate(self.model1_features.items()):
                # feat1 = feat1[:, :1]
                feat1 = feat1[:, : self.model1.num_prefix_tokens].flatten(1)
                X = feat1.flatten(1)
                K = X @ X.t()
                self.hsic_matrix[i, :, 0] += self._HSIC(K, K) / num_batches

                for j, (name2, feat2) in enumerate(self.model2_features.items()):
                    # feat2 = feat2[:, :1]
                    feat2 = feat2[:, : self.model2.num_prefix_tokens].flatten(1)
                    Y = feat2.flatten(1)
                    L = Y @ Y.t()
                    self.hsic_matrix[i, j, 1] += self._HSIC(K, L) / num_batches
                    self.hsic_matrix[i, j, 2] += self._HSIC(L, L) / num_batches

        self.hsic_matrix = self.hsic_matrix[:, :, 1] / (
            self.hsic_matrix[:, :, 0].clamp(min=1e-12).sqrt()
            * self.hsic_matrix[:, :, 2].clamp(min=1e-12).sqrt()
        )
        self.hsic_matrix = self.hsic_matrix.to("cpu")

        assert not torch.isnan(self.hsic_matrix).any(), "HSIC computation resulted in NANs"

# Model Loader

In [ ]:
def get_freqmae(weights_path):
    encoder = SSLEncoder(
        model_name="timm/vit_base_patch14_reg4_dinov2.lvd142m",
        algorithm="freqmae",
        pretrained=True,
        ckpt_path=weights_path,
        mask_ratio=0.0,
        img_size=224,
        patch_size=16,
        reg_tokens=4,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        drop_rate=0.0,
        pos_drop_rate=0.0,
        patch_drop_rate=0.0,
        proj_drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        standardize=True,
        dynamic_img_size=True,
        norm_layer=nn.LayerNorm,
    )
    return encoder.backbone


def get_mae(weights_path):
    encoder = SSLEncoder(
        model_name="timm/vit_base_patch14_reg4_dinov2.lvd142m",
        algorithm="mae",
        pretrained=True,
        ckpt_path=weights_path,
        mask_ratio=0.0,
        img_size=224,
        patch_size=16,
        reg_tokens=4,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        drop_rate=0.0,
        pos_drop_rate=0.0,
        patch_drop_rate=0.0,
        proj_drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        standardize=True,
        dynamic_img_size=True,
    )
    return encoder.backbone

# Data Loader

In [ ]:
dataframe = Dataframe(
    root="your_path/DATASET/RFMiD",
    image_folder_name="images",
    label_file_name="labels-duplicate_rm",
    suffix="base_proc-224x224",
).dataframe

# dataframe_dr = dataframe[dataframe["cls_DiabeticRetinopathy"] == 1]
# dataframe_amd = dataframe[dataframe["cls_Age-relatedMacularDegeneration"] == 1]
# dataframe_rvo = dataframe[
#     (dataframe["cls_BranchRetinalVeinOcclusion"] == 1)
#     | (dataframe["cls_CentralRetinalVeinOcclusion"] == 1)
# ]
dataframe_lesion = dataframe[
    (dataframe["cls_DiabeticRetinopathy"] == 1)
    | (dataframe["cls_Age-relatedMacularDegeneration"] == 1)
    | (dataframe["cls_BranchRetinalVeinOcclusion"] == 1)
    | (dataframe["cls_CentralRetinalVeinOcclusion"] == 1)
]

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
        self.transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]["filepath"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if image.max() > 1:
            image = image / 255.0

        return image


dataset = ImageDataset(dataframe)
# dataset_dr = ImageDataset(dataframe_dr)
# dataset_amd = ImageDataset(dataframe_amd)
# dataset_rvo = ImageDataset(dataframe_rvo)
dataset_lesion = ImageDataset(dataframe_lesion)

dataloader = DataLoader(dataset, batch_size=128, shuffle=False)
dataloader_lesion = DataLoader(dataset_lesion, batch_size=128, shuffle=False)
# dataloader_dr = DataLoader(dataset_dr, batch_size=256, shuffle=False)
# dataloader_amd = DataLoader(dataset_amd, batch_size=256, shuffle=False)
# dataloader_rvo = DataLoader(dataset_rvo, batch_size=256, shuffle=False)

# Model Load

In [ ]:
layer_num = 12
layer_names = [f"model.blocks.{i}" for i in range(layer_num)]

freq_mae = get_freqmae(
    "your_path/save/fundus_ssl/train/ssl-freqmae-vit_base_224_16-42/runs/2025-04-24_14-08-30/checkpoints/encoder_decoder-step_0126280.ckpt"
)
cka_freq_mae = CKA_FreqMAE(
    freq_mae,
    deepcopy(freq_mae),
    model1_name="ViT",  # good idea to provide names to avoid confusion
    model2_name="Masked ViT",
    model1_layers=layer_names,  # List of layers to extract features from
    model2_layers=layer_names,  # extracts all layer features by default
    device="cuda",
)

mae = get_freqmae(
    "your_path/save/fundus_ssl/train/ssl-mae-vit_base_224_16-42/runs/2025-04-04_20-41-19/checkpoints/encoder_decoder-step_0109880.ckpt"
)
cka_mae = CKA_FreqMAE(
    mae,
    deepcopy(mae),
    model1_name="ViT",  # good idea to provide names to avoid confusion
    model2_name="Masked ViT",
    model1_layers=layer_names,  # List of layers to extract features from
    model2_layers=layer_names,  # extracts all layer features by default
    device="cuda",
)

mae = get_mae(
    "your_path/save/fundus_ssl/train/ssl-mae-vit_base_224_16-42/runs/2025-04-04_20-41-19/checkpoints/encoder_decoder-step_0109880.ckpt"
)
cka_random = CKA_MAE(
    mae,
    deepcopy(mae),
    model1_name="ViT",  # good idea to provide names to avoid confusion
    model2_name="Masked ViT",
    model1_layers=layer_names,  # List of layers to extract features from
    model2_layers=layer_names,  # extracts all layer features by default
    device="cuda",
)

In [ ]:
save_file_names = ["cka_lesion_prefix_tokens", "cka_prefix_tokens"]
for idx, dataloader in enumerate([dataloader_lesion, dataloader]):
    # Create empty DataFrames to store results
    res_df_freq_mae_hf = pd.DataFrame()
    res_df_freq_mae_lf = pd.DataFrame()
    res_df_mae_hf = pd.DataFrame()
    res_df_mae_lf = pd.DataFrame()
    res_df_random = pd.DataFrame()

    # Process in smaller batches and clear GPU cache between operations

    # FreqMAE high frequency masking
    res = defaultdict(list)
    for i in range(11):
        torch.cuda.empty_cache()  # Clear GPU cache
        high_freq_ratio = i * 0.1
        cka_freq_mae.compare(
            dataloader, high_freq_keep_ratio=1 - high_freq_ratio, low_freq_keep_ratio=1
        )
        linear_cka = torch.diag(cka_freq_mae.hsic_matrix)
        res["Mask Ratio"].append(round(high_freq_ratio, 1))
        for j in range(len(linear_cka)):
            res[f"Block {j+1}"].append(linear_cka[j].item())
    res_df_freq_mae_hf = pd.DataFrame(res)

    # FreqMAE low frequency masking
    res = defaultdict(list)
    for i in range(11):
        torch.cuda.empty_cache()  # Clear GPU cache
        low_freq_ratio = i * 0.1
        cka_freq_mae.compare(
            dataloader, high_freq_keep_ratio=1, low_freq_keep_ratio=1 - low_freq_ratio
        )
        linear_cka = torch.diag(cka_freq_mae.hsic_matrix)
        res["Mask Ratio"].append(round(low_freq_ratio, 1))
        for j in range(len(linear_cka)):
            res[f"Block {j+1}"].append(linear_cka[j].item())
    res_df_freq_mae_lf = pd.DataFrame(res)

    # Save results to Excel
    with pd.ExcelWriter(f"{save_file_names[idx]}.xlsx", mode="w") as writer:
        res_df_freq_mae_hf.to_excel(writer, sheet_name="freq_mae_high_freq_masking", index=False)
        res_df_freq_mae_lf.to_excel(writer, sheet_name="freq_mae_low_freq_masking", index=False)

    # MAE high frequency masking
    res = defaultdict(list)
    for i in range(11):
        torch.cuda.empty_cache()  # Clear GPU cache
        high_freq_ratio = i * 0.1
        cka_mae.compare(
            dataloader, high_freq_keep_ratio=1 - high_freq_ratio, low_freq_keep_ratio=1
        )
        linear_cka = torch.diag(cka_mae.hsic_matrix)
        res["Mask Ratio"].append(round(high_freq_ratio, 1))
        for j in range(len(linear_cka)):
            res[f"Block {j+1}"].append(linear_cka[j].item())
    res_df_mae_hf = pd.DataFrame(res)

    # MAE low frequency masking
    res = defaultdict(list)
    for i in range(11):
        torch.cuda.empty_cache()  # Clear GPU cache
        low_freq_ratio = i * 0.1
        cka_mae.compare(dataloader, high_freq_keep_ratio=1, low_freq_keep_ratio=1 - low_freq_ratio)
        linear_cka = torch.diag(cka_mae.hsic_matrix)
        res["Mask Ratio"].append(round(low_freq_ratio, 1))
        for j in range(len(linear_cka)):
            res[f"Block {j+1}"].append(linear_cka[j].item())
    res_df_mae_lf = pd.DataFrame(res)

    # Save results to Excel
    with pd.ExcelWriter(f"{save_file_names[idx]}.xlsx", mode="a") as writer:
        res_df_mae_hf.to_excel(writer, sheet_name="mae_high_freq_masking", index=False)
        res_df_mae_lf.to_excel(writer, sheet_name="mae_low_freq_masking", index=False)

    # Random masking
    res = defaultdict(list)
    for i in range(10):
        torch.cuda.empty_cache()  # Clear GPU cache
        pred_ratio = i * 0.1
        cka_random.compare(dataloader, pred_ratio=1 - pred_ratio)
        linear_cka = torch.diag(cka_random.hsic_matrix)
        res["Mask Ratio"].append(round(pred_ratio, 1))
        for j in range(len(linear_cka)):
            res[f"Block {j+1}"].append(linear_cka[j].item())
    res_df_random = pd.DataFrame(res)

    # Save results to Excel
    with pd.ExcelWriter(f"{save_file_names[idx]}.xlsx", mode="a") as writer:
        res_df_random.to_excel(writer, sheet_name="random_masking", index=False)

    print(f"{save_file_names[idx]}.xlsx 파일이 저장되었습니다.")